# DKTC v4 - 최종 최적화 버전

**v3 대비 추가 전략 (5개):**
- [v4-1] LLRD (Layer-wise Learning Rate Decay): 트랜스포머 레이어별 차등 학습률
- [v4-2] FGM 적대적 훈련 (Miyato et al., 2017): 임베딩 perturbation으로 강건성 향상
- [v4-3] Label Smoothing: Focal Loss에 라벨 스무딩 적용하여 과신 방지
- [v4-4] EMA (Exponential Moving Average): 가중치 이동평균으로 안정적 수렴
- [v4-5] Weighted Ensemble + Threshold Optimization: 성능 가중 앙상블 + OOF 기반 클래스별 임계값 최적화

**기존 v3 전략:**
- [v3-1] K-Fold Ensemble (5-Fold)
- [v3-2] 합성 일반대화 경계 케이스 50개 (test 분포 기반)
- [v3-3] Multi-Model Ensemble: KcELECTRA + KcBERT
- [v3-4] Pseudo Labeling
- [v3-5] MAX_LEN 384

In [ ]:
# ============================================================
# STEP 0-1: 한국어 폰트 설치 + 런타임 재시작
# ============================================================
!apt-get install -y fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import os
os.kill(os.getpid(), 9)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

plt.figure()
plt.title('한국어 테스트')
plt.show()

In [ ]:
# ============================================================
# STEP 0-2: 라이브러리 설치 + 합성데이터 소스 다운로드
# ============================================================
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn pandas
!wget -q https://raw.githubusercontent.com/smilegate-ai/korean_smile_style_dataset/main/smilestyle_dataset.tsv -O /content/smilestyle_dataset.tsv
!wget -q https://raw.githubusercontent.com/Ludobico/KakaoChatData/KakaoChatData.csv -O /content/ChatbotData.csv
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt -O /content/nsmc_train.txt
print("설치 및 다운로드 완료!")

In [ ]:
# ============================================================
# Google Drive 마운트 + 작업 폴더 이동
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/DLthon'
os.chdir(DATA_DIR)

for f in ['train.csv', 'test.csv', 'submission.csv']:
    if os.path.exists(f):
        print(f"  [OK] {f}")
    else:
        print(f"  [!!] {f} 없음")

print(f"\n작업 폴더: {os.getcwd()}")

In [ ]:
import os, random, re, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# 클래스 매핑 + 하이퍼파라미터
# ============================================================
CLASS_NAMES = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']
CLASS2IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX2CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = 5

MODEL_CONFIGS = [
    {'name': 'beomi/KcELECTRA-base-v2022', 'short': 'KcELECTRA'},
    {'name': 'beomi/kcbert-base',            'short': 'KcBERT'},
]

MAX_LEN = 384
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
N_FOLDS = 5
PSEUDO_THRESHOLD = 0.95

# [v4] 새로운 하이퍼파라미터
LLRD_FACTOR = 0.95       # [v4-1] 레이어별 LR 감쇠율
FGM_EPSILON = 1.0        # [v4-2] 적대적 훈련 epsilon
LABEL_SMOOTHING = 0.05   # [v4-3] 라벨 스무딩
EMA_DECAY = 0.999        # [v4-4] EMA 감쇠율

print("v4 설정 완료")
print(f"모델: {[m['short'] for m in MODEL_CONFIGS]}")
print(f"K-Fold: {N_FOLDS}, EPOCHS: {EPOCHS}, MAX_LEN: {MAX_LEN}")
print(f"v4: LLRD={LLRD_FACTOR}, FGM={FGM_EPSILON}, LS={LABEL_SMOOTHING}, EMA={EMA_DECAY}")

In [ ]:
# ============================================================
# STEP 1: 데이터 로드
# ============================================================
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('submission.csv')

print(f"Train: {len(train_df)}개, Test: {len(test_df)}개")
print(f"클래스 분포:\n{train_df['class'].value_counts()}")
print(f"\n>> '일반 대화' 0개 → STEP 2에서 합성")

In [ ]:
# ============================================================
# STEP 2: 합성 일반대화 (5개 소스 + 경계 케이스)
# ============================================================
THREAT_KEYWORDS = [
    '죽여', '죽일', '찔러', '칼로', '패줄', '두들겨', '불질러',
    '협박', '신고', '경찰', '감옥', '고소', '소송',
    '돈 내놔', '송금', '이자', '빚', '갚아',
    '해고', '짤리', '사직서', '퇴사', '상사',
    '따돌', '왕따', '무시', '괴롭'
]

def contains_threat(text):
    return any(kw in str(text) for kw in THREAT_KEYWORDS)

normal_samples = []

# -- 소스 1: SmileStyle (400개) --
print("소스 1: SmileStyle")
try:
    smile_df = pd.read_csv('/content/smilestyle_dataset.tsv', sep='\t')
    target_cols = [c for c in smile_df.columns
                   if any(kw in c.lower() for kw in ['informal', 'chat', '반말', 'casual'])]
    if not target_cols:
        target_cols = [smile_df.columns[-1]]
    smile_texts = []
    for col in target_cols:
        smile_texts.extend(smile_df[col].dropna().tolist())
    smile_filtered = [t for t in smile_texts if not contains_threat(t) and 20 < len(str(t)) < 500]
    random.shuffle(smile_filtered)
    smile_convs = []
    for i in range(0, len(smile_filtered) - 2, 3):
        conv = ' '.join(smile_filtered[i:i+3])
        if 50 < len(conv) < 500:
            smile_convs.append(conv)
    normal_samples.extend(smile_convs[:400])
    print(f"  -> {min(400, len(smile_convs))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# -- 소스 2: kor_unsmile (500개) --
print("소스 2: kor_unsmile")
try:
    unsmile_ds = load_dataset('smilegate-ai/kor_unsmile', split='train')
    unsmile_df = unsmile_ds.to_pandas()
    if 'clean' in unsmile_df.columns:
        clean_texts = unsmile_df[unsmile_df['clean'] == 1]['문장'].tolist()
    else:
        label_cols = [c for c in unsmile_df.columns if c not in ['문장', 'clean']]
        clean_mask = unsmile_df[label_cols].sum(axis=1) == 0
        clean_texts = unsmile_df[clean_mask]['문장'].tolist()
    clean_filtered = [t for t in clean_texts if not contains_threat(t) and 10 < len(str(t)) < 300]
    random.shuffle(clean_filtered)
    unsmile_convs = []
    for i in range(0, len(clean_filtered) - 3, 4):
        conv = ' '.join(clean_filtered[i:i+4])
        if 50 < len(conv) < 500:
            unsmile_convs.append(conv)
    normal_samples.extend(unsmile_convs[:500])
    print(f"  -> {min(500, len(unsmile_convs))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# -- 소스 3: KakaoChatData (300개) --
print("소스 3: KakaoChatData")
try:
    kakao_df = pd.read_csv('/content/ChatbotData.csv')
    kakao_convs = []
    for _, row in kakao_df.iterrows():
        q = str(row.get('Q', row.iloc[0]))
        a = str(row.get('A', row.iloc[1]))
        conv = f"{q} {a}"
        if not contains_threat(conv) and 20 < len(conv) < 500:
            kakao_convs.append(conv)
    random.shuffle(kakao_convs)
    kakao_multi = []
    for i in range(0, len(kakao_convs) - 2, 3):
        conv = ' '.join(kakao_convs[i:i+3])
        if 80 < len(conv) < 500:
            kakao_multi.append(conv)
    normal_samples.extend(kakao_multi[:300])
    print(f"  -> {min(300, len(kakao_multi))}개 수집")
except Exception as e:
    print(f"  오류: {e} (건너뜀)")

# -- 소스 4: NSMC (100개) --
print("소스 4: NSMC")
try:
    nsmc_df = pd.read_csv('/content/nsmc_train.txt', sep='\t')
    daily_keywords = ['재밌', '좋았', '최고', '감동', '웃기', '대박', '꿀잼', '힐링', '따뜻', '행복', '사랑']
    positive = nsmc_df[nsmc_df['label'] == 1]['document'].dropna().tolist()
    daily_reviews = [t for t in positive
                     if any(kw in str(t) for kw in daily_keywords)
                     and not contains_threat(t) and 15 < len(str(t)) < 200]
    random.shuffle(daily_reviews)
    nsmc_convs = []
    for i in range(0, len(daily_reviews) - 4, 5):
        conv = ' '.join(daily_reviews[i:i+5])
        if 80 < len(conv) < 500:
            nsmc_convs.append(conv)
    normal_samples.extend(nsmc_convs[:100])
    print(f"  -> {min(100, len(nsmc_convs))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# -- 소스 5: 경계 케이스 (50개, test 분포 기반) --
print("소스 5: 경계 케이스 (test 분포 기반)")
boundary_cases = [
    # 패턴 A: 협박처럼 보이지만 일반 대화 (15개)
    "야 죽을래 ㅋㅋ 아 진짜 웃겨서 죽겠다 아 배아파 ㅋㅋㅋ 진짜 미쳤어 너 개그맨 해라",
    "야 너 진짜 맞을래 ㅋㅋ 아 왜 그런 말을 해서 웃기게 만들어 아 진짜 복근 생기겠다",
    "죽여버린다 ㅋㅋ 아 이 게임 왜 이렇게 어려워 보스 죽여버리고 싶다 진짜",
    "패버리고 싶다 ㅋㅋ 누구를 이 게임 캐릭터 진짜 짜증나 아 다시 해야지",
    "칼로 자르고 싶다 뭘 이 케이크 너무 예뻐서 자르기 아까운데 먹어야지",
    "야 한대 맞을래 ㅋㅋㅋ 농담이야 근데 진짜 왜 그런 얘기를 해 아 웃겨",
    "미쳤어 진짜 ㅋㅋ 이 짤 봤어 와 진짜 웃겨서 죽는줄 알았어 보내줄까",
    "너 진짜 미쳤다 ㅋㅋ 이걸 어떻게 생각해내 와 천재 아니야 대단하다 진짜",
    "야 너 오늘 죽었다 ㅋㅋ 왜 노래방에서 내 노래 뺏어서 불렀잖아 다음엔 내가 먼저",
    "때려치우고 싶다 뭘 회사 오늘 진짜 힘들었어 야 치킨 먹자 나 오늘 자격 있어",
    "경찰 부를거야 ㅋㅋ 왜 너 방 너무 더러워서 환경오염 신고해야 할 것 같아",
    "고소할거야 진짜 ㅋㅋ 왜 네가 나한테 이렇게 맛있는 걸 안 알려줬어 배신자",
    "불질러버리고 싶다 뭘 이 과제 너무 많아서 다 태워버리고 싶어 ㅋㅋ 농담이야",
    "찔러버린다 ㅋㅋ 뭘 이 젤리 포크로 찔러서 먹어야 하는데 손으로 먹었어",
    "감옥에 넣어야 해 ㅋㅋ 누구를 이렇게 늦게 연락하는 너를 지각죄로 가둬야지",
    # 패턴 B: 기타 괴롭힘처럼 보이지만 일반 대화 (15개)
    "야 너 왜 혼자 밥 먹어 같이 먹자 아 몰랐어 미안 내일부터 같이 가자",
    "쟤 좀 이상하지 않아 뭐가 아 그냥 오늘 옷 되게 특이하게 입었더라 멋있어",
    "왕따 당하는 기분이야 ㅋㅋ 왜 아 오늘 점심 메뉴 나만 몰랐어 다들 알려줘",
    "무시하지 마 ㅋㅋ 내 말 좀 들어봐 이번 여행 계획 진짜 좋거든 어디냐면",
    "따돌리는거야 ㅋㅋ 왜 단톡방에 나만 안 넣었어 아 새로 만든거야 초대해줘",
    "너네 나 없이 놀았지 ㅋㅋ 사진 봤어 아 미안 갑자기 된거야 다음엔 꼭 같이 가자",
    "괴롭히지 마 ㅋㅋ 야 자꾸 내 별명 부르지 마 그거 초등학교 때 거잖아 창피해",
    "소외감 느껴 ㅋㅋ 왜 너네 다 커플이라 나만 혼자야 아 나도 소개팅 시켜줘",
    "야 쟤 왜 저래 아 그냥 원래 좀 조용한 애야 착해 알고 보면 진짜 재밌어",
    "나만 빼고 다 아는거야 뭘 아 그 유행어 나만 모르나 알려줘 뭔데",
    "혼자 있고 싶어 왜 아 그냥 오늘 좀 피곤해서 내일 만나자 그래 푹 쉬어",
    "무시당한 기분이야 ㅋㅋ 왜 아 내가 추천한 맛집을 아무도 안 가줘서 나만 좋아하나",
    "같이 안 놀아줘 ㅋㅋ 누가 우리 강아지가 다른 강아지 싫어해서 혼자 놀아 귀여워",
    "차별하는거 아니야 ㅋㅋ 왜 아 치킨 양념만 시키지 말고 후라이드도 시키자 나는 후라이드파",
    "아무도 안 좋아해 ㅋㅋ 뭘 이 맛집을 아무도 모르더라 나만 알고 있어 데려갈까",
    # 패턴 C: 갈취처럼 보이지만 일반 대화 (10개)
    "돈 내놔 ㅋㅋ 밥값 네가 쏜다며 아 맞다 내가 쏜다고 했지 ㅋㅋ 어디 갈까",
    "야 천원만 빌려줘 자판기 커피 마시고 싶은데 지갑을 놓고 왔어 내일 바로 갚을게",
    "야 그거 빌려줘 뭘 충전기 배터리 없어서 잠깐만 쓸게 고마워",
    "밥 사라 ㅋㅋ 야 오늘 내 생일인데 당연히 네가 사야지 어디 갈까",
    "용돈 다 썼어 ㅋㅋ 야 오늘 커피 한잔만 사줘 다음에 내가 쏠게 진짜로",
    "야 택시비 좀 보태줘 3천원만 있으면 되는데 집에 가야 해 내일 바로 보내줄게",
    "야 이거 줘 뭐 그 펜 좀 잠깐만 쓸게 아 고마워 나중에 돌려줄게",
    "돈 있어 얼마 만원만 있으면 되는데 같이 밥 먹으러 가자 내가 부족한 부분 낼게",
    "야 담배 한 개비 줘봐 아 나 오늘 스트레스 받아서 한 대만 ㅋㅋ 고마워 내일 사줄게",
    "이거 나 좀 줘 뭐 이 과자 맛있어 보여서 하나만 줘봐 오 진짜 맛있다",
    # 패턴 D: 직장 괴롭힘처럼 보이지만 일반 대화 (10개)
    "오늘 야근이야 또 아 진짜 힘들다 그래도 이번 프로젝트 끝나면 좀 쉴 수 있겠지",
    "상사가 또 일 줬어 근데 뭐 그래도 인정해주니까 열심히 해야지 파이팅",
    "퇴사하고 싶다 ㅋㅋ 아 농담이야 월급날이니까 참는거지 오늘 뭐 먹을까",
    "야 우리 부장님 또 회식 잡았대 아 귀찮다 그래도 고기니까 ㅋㅋ 가자",
    "아 오늘 진짜 일 많다 죽겠어 ㅋㅋ 그래도 퇴근하면 치맥이다 버텨보자",
    "팀장님이 또 수정해달래 세번째야 근데 뭐 덕분에 더 좋아지긴 했어 감사하지",
    "인사팀에서 면담하자고 했어 뭐래 아 그냥 만족도 조사래 놀랐잖아 ㅋㅋ",
    "해고당할뻔 했어 ㅋㅋ 왜 아 늦잠자서 지각했는데 부장님이 웃으면서 넘어가줬어 휴",
    "사직서 쓸뻔 했다 ㅋㅋ 왜 프린터가 안 돼서 30분 싸웠어 결국 고쳤어",
    "회의 또 해 진짜 오늘만 세번째야 그래도 뭐 좋은 아이디어 나왔으니까 괜찮아",
]
normal_samples.extend(boundary_cases)
print(f"  -> {len(boundary_cases)}개 추가")
print(f"    협박 경계: 15, 기타괴롭힘 경계: 15, 갈취 경계: 10, 직장괴롭힘 경계: 10")
print(f"\n총 합성 일반대화: {len(normal_samples)}개")

In [ ]:
# ============================================================
# 합성 데이터 통합 + 전처리
# ============================================================
normal_df = pd.DataFrame({
    'idx': [f'n_{i:03d}' for i in range(len(normal_samples))],
    'class': '일반 대화',
    'conversation': normal_samples
})

train_full = pd.concat([train_df[['idx', 'class', 'conversation']], normal_df],
                       ignore_index=True)
print(f"통합 train: {len(train_full)}개")
print(train_full['class'].value_counts())

def preprocess(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^가-힣a-zA-Z0-9ㄱ-ㅎㅏ-ㅣ\s,.!?~ㅋㅎㅠㅜ]', '', text)
    return text.strip()

train_full['conversation'] = train_full['conversation'].apply(preprocess)
test_df['conversation'] = test_df['conversation'].apply(preprocess)
train_full['label'] = train_full['class'].map(CLASS2IDX).astype(int)

print(f"\n전처리 완료. 라벨 분포:")
print(train_full['label'].value_counts().sort_index())

In [ ]:
# ============================================================
# STEP 4: Dataset, Loss, FGM, EMA, LLRD 정의
# ============================================================
class DKTCDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    # Focal Loss (Lin et al., 2017) + [v4-3] Label Smoothing
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.alpha,
            reduction='none', label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-F.cross_entropy(inputs, targets, reduction='none'))
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss


def compute_rdrop_loss(logits1, logits2, labels, loss_fn, alpha=0.7):
    # R-Drop (Liang et al., 2021)
    loss1 = loss_fn(logits1, labels)
    loss2 = loss_fn(logits2, labels)
    ce_loss = (loss1 + loss2) / 2
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    kl_loss = (
        F.kl_div(p, q.exp(), reduction='batchmean') +
        F.kl_div(q, p.exp(), reduction='batchmean')
    ) / 2
    return ce_loss + alpha * kl_loss


class FGM:
    # [v4-2] Fast Gradient Method (Miyato et al., 2017)
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()
                if param.grad is not None:
                    norm = torch.norm(param.grad)
                    if norm != 0 and not torch.isnan(norm):
                        r_at = self.epsilon * param.grad / norm
                        param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                if name in self.backup:
                    param.data = self.backup[name]
        self.backup = {}


class EMA:
    # [v4-4] Exponential Moving Average
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_avg = (1 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_avg.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}


def get_llrd_optimizer(model, lr=2e-5, weight_decay=0.01, llrd_factor=0.95):
    # [v4-1] Layer-wise Learning Rate Decay
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    if hasattr(model, 'electra'):
        backbone = model.electra
    elif hasattr(model, 'bert'):
        backbone = model.bert
    else:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    layers = [backbone.embeddings] + list(backbone.encoder.layer)
    num_layers = len(layers)

    opt_params = []
    for i, layer in enumerate(layers):
        layer_lr = lr * (llrd_factor ** (num_layers - 1 - i))
        opt_params.extend([
            {'params': [p for n, p in layer.named_parameters()
                       if not any(nd in n for nd in no_decay) and p.requires_grad],
             'lr': layer_lr, 'weight_decay': weight_decay},
            {'params': [p for n, p in layer.named_parameters()
                       if any(nd in n for nd in no_decay) and p.requires_grad],
             'lr': layer_lr, 'weight_decay': 0.0},
        ])

    # Classifier head: full lr
    cls_decay, cls_no_decay = [], []
    for name, param in model.named_parameters():
        if 'classifier' in name and param.requires_grad:
            if any(nd in name for nd in no_decay):
                cls_no_decay.append(param)
            else:
                cls_decay.append(param)
    if cls_decay:
        opt_params.append({'params': cls_decay, 'lr': lr, 'weight_decay': weight_decay})
    if cls_no_decay:
        opt_params.append({'params': cls_no_decay, 'lr': lr, 'weight_decay': 0.0})

    return torch.optim.AdamW(opt_params)

print("v4 클래스 정의 완료: Dataset, FocalLoss, R-Drop, FGM, EMA, LLRD")

In [ ]:
# ============================================================
# STEP 5: 학습/검증/예측 함수 (v4)
# ============================================================
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_fn,
                    fgm=None, ema=None, use_rdrop=True, rdrop_alpha=0.7):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in dataloader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if 'token_type_ids' in batch:
            model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        # Forward (with R-Drop)
        if use_rdrop:
            outputs1 = model(**model_kwargs)
            outputs2 = model(**model_kwargs)
            loss = compute_rdrop_loss(
                outputs1.logits, outputs2.logits, labels,
                loss_fn, alpha=rdrop_alpha
            )
            logits = outputs1.logits
        else:
            outputs = model(**model_kwargs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        # [v4-2] FGM adversarial training
        if fgm is not None:
            fgm.attack()
            adv_out = model(**model_kwargs)
            adv_loss = loss_fn(adv_out.logits, labels)
            adv_loss.backward()
            fgm.restore()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        # [v4-4] EMA update
        if ema is not None:
            ema.update()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


def evaluate(model, dataloader, loss_fn):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

            outputs = model(**model_kwargs)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1, all_preds, all_labels


def predict_proba(model, dataloader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
            outputs = model(**model_kwargs)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs)


def optimize_thresholds(oof_probs, oof_labels, num_classes=5):
    # [v4-5] OOF 기반 클래스별 threshold 최적화
    best_thresholds = np.ones(num_classes)
    base_preds = np.argmax(oof_probs, axis=1)
    best_f1 = f1_score(oof_labels, base_preds, average='macro')
    print(f"  기본 OOF F1: {best_f1:.4f}")

    for cls in range(num_classes):
        cls_best_t = 1.0
        for t in np.arange(0.5, 2.01, 0.05):
            adjusted = oof_probs.copy()
            adjusted[:, cls] *= t
            preds = np.argmax(adjusted, axis=1)
            f1 = f1_score(oof_labels, preds, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                cls_best_t = t
        best_thresholds[cls] = cls_best_t

    print(f"  최적화 후 OOF F1: {best_f1:.4f}")
    print(f"  Thresholds: {dict(zip(CLASS_NAMES, [f'{t:.2f}' for t in best_thresholds]))}")
    return best_thresholds

print("v4 학습/평가/예측 함수 정의 완료")

In [ ]:
# ============================================================
# STEP 6: K-Fold x Multi-Model 학습
# v3: K-Fold, Multi-Model | v4: LLRD, FGM, Label Smoothing, EMA
# ============================================================
all_model_probs = []
all_fold_results = []
all_oof_probs = []  # per-model OOF arrays
oof_labels = train_full['label'].values.copy()

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg['name']
    model_short = model_cfg['short']
    print(f"\n{'='*60}")
    print(f"  모델: {model_short} ({model_name})")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    test_dataset = DKTCDataset(
        test_df['conversation'].values, labels=None,
        tokenizer=tokenizer, max_len=MAX_LEN
    )
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    model_oof = np.zeros((len(train_full), NUM_CLASSES))

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_full, train_full['label'])):
        print(f"\n  --- {model_short} Fold {fold+1}/{N_FOLDS} ---")

        fold_train = train_full.iloc[train_idx]
        fold_val = train_full.iloc[val_idx]

        train_dataset = DKTCDataset(
            fold_train['conversation'].values, fold_train['label'].values,
            tokenizer, MAX_LEN
        )
        val_dataset = DKTCDataset(
            fold_val['conversation'].values, fold_val['label'].values,
            tokenizer, MAX_LEN
        )
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

        # Class weights
        label_counts = fold_train['label'].value_counts().sort_index()
        total = len(fold_train)
        class_weights = torch.tensor(
            [total / (NUM_CLASSES * count) for count in label_counts.values],
            dtype=torch.float32
        ).to(DEVICE)

        # [v4-3] Label Smoothing in FocalLoss
        loss_fn = FocalLoss(
            alpha=class_weights, gamma=2.0,
            label_smoothing=LABEL_SMOOTHING
        ).to(DEVICE)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=NUM_CLASSES
        ).to(DEVICE)

        # [v4-1] LLRD optimizer
        optimizer = get_llrd_optimizer(
            model, lr=LR, weight_decay=WEIGHT_DECAY, llrd_factor=LLRD_FACTOR
        )
        total_steps = len(train_loader) * EPOCHS
        warmup_steps = int(total_steps * WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        # [v4-2] FGM + [v4-4] EMA
        fgm = FGM(model, epsilon=FGM_EPSILON)
        ema = EMA(model, decay=EMA_DECAY)

        best_val_f1 = 0
        best_state = None

        for epoch in range(EPOCHS):
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, scheduler, loss_fn,
                fgm=fgm, ema=ema, use_rdrop=True, rdrop_alpha=0.7
            )
            # Evaluate with EMA weights
            ema.apply_shadow()
            val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, loss_fn)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ema.restore()

            print(f"    Ep {epoch+1}/{EPOCHS} | "
                  f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}"
                  f"{'  *' if val_f1 >= best_val_f1 else ''}")

        print(f"    >> Best Val F1: {best_val_f1:.4f}")

        # Load best state (EMA weights) for prediction
        model.load_state_dict(best_state)
        model.to(DEVICE)

        # Test predictions
        fold_probs = predict_proba(model, test_loader)
        all_model_probs.append(fold_probs)

        # OOF predictions for threshold optimization
        val_probs = predict_proba(model, val_loader)
        model_oof[val_idx] = val_probs

        all_fold_results.append({
            'model': model_short,
            'fold': fold + 1,
            'best_val_f1': best_val_f1
        })

        del model, fgm, ema
        gc.collect()
        torch.cuda.empty_cache()

    all_oof_probs.append(model_oof)

print("\n학습 완료!")

In [ ]:
# ============================================================
# STEP 7: [v4-5] Weighted Ensemble + Threshold Optimization
# ============================================================
print(f"\n{'='*60}")
print(f"  [v4-5] Weighted Ensemble + Threshold Optimization")
print(f"{'='*60}")

for r in all_fold_results:
    print(f"  {r['model']} Fold {r['fold']}: Val F1 = {r['best_val_f1']:.4f}")

avg_val_f1 = np.mean([r['best_val_f1'] for r in all_fold_results])
print(f"\n  평균 Val F1: {avg_val_f1:.4f}")

# Performance-weighted ensemble
weights = np.array([r['best_val_f1'] for r in all_fold_results])
weights = weights / weights.sum()
print(f"  모델 가중치: {[f'{w:.3f}' for w in weights]}")

weighted_probs = np.zeros_like(all_model_probs[0])
for w, probs in zip(weights, all_model_probs):
    weighted_probs += w * probs

# OOF threshold optimization
avg_oof = np.mean(all_oof_probs, axis=0)
print(f"\n  Threshold 최적화 (OOF 기반):")
thresholds = optimize_thresholds(avg_oof, oof_labels, NUM_CLASSES)

# Preview ensemble predictions with thresholds
adjusted_preview = weighted_probs.copy()
for cls in range(NUM_CLASSES):
    adjusted_preview[:, cls] *= thresholds[cls]
preview_preds = np.argmax(adjusted_preview, axis=1)

print(f"\n  앙상블 예측 분포 (threshold 적용):")
preview_counts = Counter(preview_preds)
for label_idx in sorted(preview_counts.keys()):
    print(f"    {IDX2CLASS[label_idx]}: {preview_counts[label_idx]}개")

In [ ]:
# ============================================================
# STEP 8: Pseudo Labeling + 재학습 (v4 전략 적용)
# ============================================================
print(f"\n{'='*60}")
print(f"  [v3-4] Pseudo Labeling (threshold={PSEUDO_THRESHOLD})")
print(f"{'='*60}")

max_probs = np.max(weighted_probs, axis=1)
confident_mask = max_probs >= PSEUDO_THRESHOLD
pseudo_labels = np.argmax(weighted_probs, axis=1)[confident_mask]

print(f"  전체 test: {len(test_df)}개")
print(f"  고확신 샘플: {confident_mask.sum()}개 ({confident_mask.sum()/len(test_df)*100:.1f}%)")

if confident_mask.sum() > 0:
    pseudo_counts = Counter(pseudo_labels)
    for label_idx in sorted(pseudo_counts.keys()):
        print(f"    {IDX2CLASS[label_idx]}: {pseudo_counts[label_idx]}개")

    pseudo_df = pd.DataFrame({
        'idx': test_df[confident_mask]['idx'].values,
        'class': [IDX2CLASS[l] for l in pseudo_labels],
        'conversation': test_df[confident_mask]['conversation'].values,
        'label': pseudo_labels
    })
    train_pseudo = pd.concat([train_full, pseudo_df], ignore_index=True)
    print(f"\n  train + pseudo: {len(train_pseudo)}개")

    print(f"\n  Pseudo 재학습 (KcELECTRA + v4 전략)...")
    tokenizer_p = AutoTokenizer.from_pretrained(MODEL_CONFIGS[0]['name'])

    p_train, p_val = train_test_split(
        train_pseudo, test_size=0.1, stratify=train_pseudo['label'], random_state=SEED
    )
    p_train_ds = DKTCDataset(p_train['conversation'].values, p_train['label'].values, tokenizer_p, MAX_LEN)
    p_val_ds = DKTCDataset(p_val['conversation'].values, p_val['label'].values, tokenizer_p, MAX_LEN)
    p_train_loader = DataLoader(p_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    p_val_loader = DataLoader(p_val_ds, batch_size=BATCH_SIZE)

    p_counts = p_train['label'].value_counts().sort_index()
    p_total = len(p_train)
    p_weights = torch.tensor(
        [p_total / (NUM_CLASSES * c) for c in p_counts.values],
        dtype=torch.float32
    ).to(DEVICE)
    p_loss_fn = FocalLoss(alpha=p_weights, gamma=2.0, label_smoothing=LABEL_SMOOTHING).to(DEVICE)

    p_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CONFIGS[0]['name'], num_labels=NUM_CLASSES
    ).to(DEVICE)
    p_optimizer = get_llrd_optimizer(p_model, lr=LR, weight_decay=WEIGHT_DECAY, llrd_factor=LLRD_FACTOR)
    p_total_steps = len(p_train_loader) * EPOCHS
    p_scheduler = get_linear_schedule_with_warmup(
        p_optimizer, int(p_total_steps * WARMUP_RATIO), p_total_steps
    )
    p_fgm = FGM(p_model, epsilon=FGM_EPSILON)
    p_ema = EMA(p_model, decay=EMA_DECAY)

    best_p_f1 = 0
    best_p_state = None

    for epoch in range(EPOCHS):
        t_loss, t_acc, t_f1 = train_one_epoch(
            p_model, p_train_loader, p_optimizer, p_scheduler, p_loss_fn,
            fgm=p_fgm, ema=p_ema, use_rdrop=True
        )
        p_ema.apply_shadow()
        v_loss, v_acc, v_f1, _, _ = evaluate(p_model, p_val_loader, p_loss_fn)
        if v_f1 > best_p_f1:
            best_p_f1 = v_f1
            best_p_state = {k: v.cpu().clone() for k, v in p_model.state_dict().items()}
        p_ema.restore()

        print(f"    Pseudo Ep {epoch+1}/{EPOCHS} | "
              f"Train F1: {t_f1:.4f} | Val F1: {v_f1:.4f}"
              f"{'  *' if v_f1 >= best_p_f1 else ''}")

    print(f"    >> Best Pseudo Val F1: {best_p_f1:.4f}")

    p_model.load_state_dict(best_p_state)
    p_model.to(DEVICE)
    test_ds_p = DKTCDataset(test_df['conversation'].values, labels=None, tokenizer=tokenizer_p, max_len=MAX_LEN)
    test_loader_p = DataLoader(test_ds_p, batch_size=BATCH_SIZE)
    pseudo_probs = predict_proba(p_model, test_loader_p)

    # Final blend + threshold
    final_probs = 0.4 * weighted_probs + 0.6 * pseudo_probs
    for cls in range(NUM_CLASSES):
        final_probs[:, cls] *= thresholds[cls]
    final_preds = np.argmax(final_probs, axis=1)

    del p_model, p_fgm, p_ema
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("  고확신 샘플 없음 -> Pseudo Labeling 건너뜀")
    final_probs = weighted_probs.copy()
    for cls in range(NUM_CLASSES):
        final_probs[:, cls] *= thresholds[cls]
    final_preds = np.argmax(final_probs, axis=1)

In [ ]:
# ============================================================
# STEP 9: 제출파일 생성
# ============================================================
submission_df['class'] = final_preds.astype(int)
submission_df.to_csv('submission_v4.csv', index=False)

print("submission_v4.csv 저장 완료!")
print(f"\n예측 분포:")
final_counts = Counter(final_preds)
for label_idx in sorted(final_counts.keys()):
    print(f"  {IDX2CLASS[label_idx]}: {final_counts[label_idx]}개")

normal_count = sum(1 for p in final_preds if p == 4)
print(f"\n일반 대화: {normal_count}개 ({normal_count/len(final_preds)*100:.1f}%)")

In [ ]:
# ============================================================
# STEP 10: 결과 시각화
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Fold별 Val F1
ax = axes[0]
for model_cfg in MODEL_CONFIGS:
    ms = model_cfg['short']
    fold_f1s = [r['best_val_f1'] for r in all_fold_results if r['model'] == ms]
    x_labels = [f"{ms}\nF{i+1}" for i in range(len(fold_f1s))]
    ax.bar(x_labels, fold_f1s, alpha=0.7, label=ms)
ax.set_ylabel('Val F1')
ax.set_title('K-Fold Val F1 by Model (v4)')
ax.legend()
ax.set_ylim(0.85, 1.0)

# (b) 최종 예측 분포
ax = axes[1]
labels_list = [IDX2CLASS[i] for i in range(NUM_CLASSES)]
counts_list = [final_counts.get(i, 0) for i in range(NUM_CLASSES)]
colors = ['#e74c3c', '#e67e22', '#3498db', '#9b59b6', '#2ecc71']
ax.barh(labels_list, counts_list, color=colors)
ax.set_title('최종 예측 분포 (v4)')
ax.set_xlabel('개수')
for i, v in enumerate(counts_list):
    ax.text(v + 2, i, str(v), va='center')

plt.tight_layout()
plt.savefig('v4_results.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 11: 프로젝트 정리
# ============================================================
print("=" * 60)
print("  DKTC v4 프로젝트 정리")
print("=" * 60)
print(f"\n  [v3-1] K-Fold: {N_FOLDS}-Fold x {len(MODEL_CONFIGS)} models")
print(f"  [v3-2] 경계 케이스: {len(boundary_cases)}개 (test 분포 기반)")
print(f"  [v3-3] Multi-Model: {[m['short'] for m in MODEL_CONFIGS]}")
print(f"  [v3-4] Pseudo Labeling: threshold={PSEUDO_THRESHOLD}")
print(f"  [v3-5] MAX_LEN: {MAX_LEN}")
print(f"\n  [v4-1] LLRD: factor={LLRD_FACTOR}")
print(f"  [v4-2] FGM: epsilon={FGM_EPSILON}")
print(f"  [v4-3] Label Smoothing: {LABEL_SMOOTHING}")
print(f"  [v4-4] EMA: decay={EMA_DECAY}")
print(f"  [v4-5] Weighted Ensemble + Threshold Opt: {thresholds}")
print(f"\n  평균 Val F1: {avg_val_f1:.4f}")
print(f"  제출파일: submission_v4.csv")
print(f"\n  완료!")

In [ ]:
from google.colab import files
files.download('submission_v4.csv')